In [0]:
# Import paths configuration
import sys
sys.path.append('/Workspace/Users/pawanvirat32@gmail.com/MatchPulse')

from config.paths import MATCHES_RAW, MATCHES_BRONZE

print(f"Reading matches from: {MATCHES_RAW}")
print(f"Will write to: {MATCHES_BRONZE}")

In [0]:
# Read all match JSON files from S3
# The files are organized as: matches/{competition_id}/{season_id}.json
# Each JSON file contains an array of match objects

from pyspark.sql.functions import regexp_extract, col

# Read JSON files recursively
df_raw = spark.read.option("multiLine", "true").json(f"{MATCHES_RAW}*/*")

# Extract competition_id and season_id from file path using _metadata.file_path (Unity Catalog compatible)
df_matches = df_raw.withColumn("_file_path", col("_metadata.file_path")) \
    .withColumn("competition_id", 
                regexp_extract(col("_file_path"), r"matches/(\d+)/", 1).cast("int")) \
    .withColumn("season_id", 
                regexp_extract(col("_file_path"), r"/(\d+)\.json", 1).cast("int")) \
    .drop("_file_path")

print(f"Total matches loaded: {df_matches.count()}")
df_matches.printSchema()

In [0]:
# Display sample matches data
display(df_matches.limit(10))

In [0]:
# Clean up existing bronze directory if it exists with incompatible format
try:
    dbutils.fs.rm(MATCHES_BRONZE, recurse=True)
    print(f"Cleaned up existing directory: {MATCHES_BRONZE}")
except Exception as e:
    print(f"No existing directory to clean: {e}")

In [0]:
# Write to bronze Delta table
# First, check if directory exists and clean if needed
import shutil
from pathlib import Path

print(f"Writing {df_matches.count()} matches to bronze Delta table...")

# Write with overwriteSchema option to handle any existing data
df_matches.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(MATCHES_BRONZE)

print(f"✓ Successfully written to {MATCHES_BRONZE}")

In [0]:
# Verify the bronze table
df_bronze = spark.read.format("delta").load(MATCHES_BRONZE)

print(f"Bronze table stats:")
print(f"  Total matches: {df_bronze.count()}")
print(f"  Competitions: {df_bronze.select('competition_id').distinct().count()}")
print(f"  Seasons: {df_bronze.select('season_id').distinct().count()}")
print(f"\nSample match IDs:")
display(df_bronze.select('match_id', 'match_date', 'home_team.home_team_name', 'away_team.away_team_name', 
                          'home_score', 'away_score', 'competition_id', 'season_id').limit(10))

In [0]:
display(df_bronze)